# Reformat

1. Missing value problem
 - missing buyer_name = "Unknown Buyer"
 - missing mainprocurementcategory = "Unknown Category"
 - trim juga sama kek training

2. Bad data
 - wajib award_value, dan award_value <= 0 gone
 - null

In [74]:
import pandas as pd
import numpy as np

data = pd.read_csv('All_data_ocd.csv')

print(f"Loaded raw data: {len(data)} rows, {len(data.columns)} columns")
print("\n--- Initial Missing Values ---")
print(data.isnull().sum().to_string())

Loaded raw data: 26034 rows, 16 columns

--- Initial Missing Values ---
ocid                          0
release_id                    0
date                          0
buyer_name                 4443
buyer_id                    207
tender_id                     0
tender_title                  0
mainProcurementCategory       0
tender_minValue               0
tender_datePublished          0
tender_status                 0
award_id                      0
award_title                   0
award_date                    0
award_value                   0
award_supplier                0


In [75]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
print("Columns:", data.columns.tolist())

Columns: ['ocid', 'release_id', 'date', 'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished', 'tender_status', 'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier']


In [76]:
# Parse dates (strip timezone for compatibility)
date_cols = ['date', 'tender_datepublished', 'award_date']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], utc=True, errors='coerce').dt.tz_localize(None)

# Cast numerics
num_cols = ['tender_minvalue', 'award_value']
for col in num_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

print("Dtypes after casting:")
print(data.dtypes.to_string())

Dtypes after casting:
ocid                                  str
release_id                            str
date                       datetime64[us]
buyer_name                            str
buyer_id                              str
tender_id                           int64
tender_title                          str
mainprocurementcategory               str
tender_minvalue                   float64
tender_datepublished       datetime64[us]
tender_status                         str
award_id                              str
award_title                           str
award_date                 datetime64[us]
award_value                       float64
award_supplier                        str


In [ ]:
before = len(data)

# duplicates
data.drop_duplicates(inplace=True)
afterDup = len(data)

print(f"Duplicate Rows removed: {before - afterDup}")

# incomplete data
beforeAw = len(data)
data.dropna(subset=['award_value', 'award_supplier'], inplace=True)
afterAw = len(data)
print(f"Awards Rows removed: {beforeAw - afterAw}")

# corr
beforeCor = len(data)
data = data[data['award_value'] > 0]
afterCor = len(data)
print(f"Corup Rows removed: {beforeCor - afterCor}")

after = len(data)
print(f"Rows removed: {before - after} ({before} -> {after})")


Duplicate Rows removed: 17356
Awards Rows removed: 0
Corup Rows removed: 0
Rows removed: 17356 (26034 -> 8678)


In [78]:
# buyer_name: try to recover from other rows sharing the same buyer_id
buyer_map = (
    data.dropna(subset=['buyer_name', 'buyer_id'])
        .drop_duplicates('buyer_id')
        .set_index('buyer_id')['buyer_name']
        .to_dict()
)
data['buyer_name'] = data['buyer_name'].fillna(data['buyer_id'].map(buyer_map))

# Remaining buyer_name nulls fill with 'Unknown Buyer'
data['buyer_name'] = data['buyer_name'].fillna('Unknown Buyer')

# mainprocurementcategory fill with 'Unknown Category'
data['mainprocurementcategory'] = data['mainprocurementcategory'].fillna('Unknown Category')

# tender_minvalue fill missing with median (reasonable estimate)
median_min = data['tender_minvalue'].median()
data['tender_minvalue'] = data['tender_minvalue'].fillna(median_min)

print("Missing values after imputation:")
print(data.isnull().sum().to_string())

Missing values after imputation:
ocid                        0
release_id                  0
date                        0
buyer_name                  0
buyer_id                   69
tender_id                   0
tender_title                0
mainprocurementcategory     0
tender_minvalue             0
tender_datepublished        0
tender_status               0
award_id                    0
award_title                 0
award_date                  0
award_value                 0
award_supplier              0


In [79]:
text_cols = ['tender_title', 'award_title', 'buyer_name', 'award_supplier',
             'mainprocurementcategory', 'tender_status']
for col in text_cols:
    data[col] = data[col].astype(str).str.strip().str.title()

print("Sample buyer_name values:")
print(data['buyer_name'].value_counts().head(10))

Sample buyer_name values:
buyer_name
Pemerintah Daerah Provinsi Dki Jakarta                7136
Unknown Buyer                                         1481
Sekretariat Kabinet                                     35
Pd Pal Jaya                                              8
Pdam Provinsi Dki Jakarta                                7
Pd Dharma Jaya                                           3
Kementerian Kesehatan                                    3
Badan Kependudukan Dan Keluarga Berencana Nasional       2
Pemerintah Daerah Kabupaten Puncak Jaya                  1
Pemerintah Daerah Kabupaten Musi Banyuasin               1
Name: count, dtype: int64


In [ ]:
data['award_year']  = data['award_date'].dt.year
data['award_month'] = data['award_date'].dt.month

# Extract year and month from tender published date
data['publish_year']  = data['tender_datepublished'].dt.year
data['publish_month'] = data['tender_datepublished'].dt.month

# Calculate days_to_award: days between tender publish and award date
data['days_to_award'] = (data['award_date'] - data['tender_datepublished']).dt.days

# Clip negative values to 0
data['days_to_award'] = data['days_to_award'].clip(lower=0)

print("New temporal features:")
print(data[['award_year', 'award_month', 'publish_year', 'publish_month', 'days_to_award']].describe())

New temporal features:
        award_year  award_month  publish_year  publish_month  days_to_award
count  8678.000000  8678.000000   8678.000000    8678.000000    8678.000000
mean   2018.130214     7.255474   2018.120304       6.516594      26.432012
std       2.578465     2.507626      2.579438       2.630946      18.603155
min    2014.000000     1.000000   2014.000000       1.000000       0.000000
25%    2015.000000     5.000000   2015.000000       4.000000      15.000000
50%    2018.000000     8.000000   2018.000000       7.000000      21.000000
75%    2019.000000     9.000000   2019.000000       9.000000      35.000000
max    2023.000000    12.000000   2023.000000      12.000000     248.000000


In [81]:
# budget_utilization_ratio: how much of the minimum tender budget was used
# Clip to avoid division by zero; cap ratio at 10x to remove extreme outliers
data['budget_utilization_ratio'] = (
    data['award_value'] / data['tender_minvalue'].replace(0, np.nan)
).clip(upper=10)
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(1.0)

print("budget_utilization_ratio stats:")
print(data['budget_utilization_ratio'].describe())

budget_utilization_ratio stats:
count    8678.000000
mean        0.876146
std         0.096232
min         0.084296
25%         0.814600
50%         0.893938
75%         0.948149
max         1.997665
Name: budget_utilization_ratio, dtype: float64


In [82]:
# Drop only true identity/noise columns
# tender_datepublished is dropped because it's been fully extracted into publish_year, publish_month, days_to_award
cols_to_drop = ['ocid', 'release_id', 'buyer_id', 'award_id', 'tender_datepublished', 'tender_id', 'award_year', 'award_month', 'publish_year', 'publish_month']

data.drop(columns=[c for c in cols_to_drop if c in data.columns], inplace=True)

print("Remaining columns:")
print(data.columns.tolist())

Remaining columns:
['date', 'buyer_name', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier', 'days_to_award', 'budget_utilization_ratio']


In [83]:
print("=" * 40)
print("FINAL DATASET SUMMARY")
print("=" * 40)
print(f"Total rows   : {len(data)}")
print(f"Total columns: {len(data.columns)}")
print("\nMissing values (should all be 0):")
print(data.isnull().sum().to_string())
print("\nColumn dtypes:")
print(data.dtypes.to_string())
print("\nSample data:")
data.head(3)

FINAL DATASET SUMMARY
Total rows   : 8678
Total columns: 12

Missing values (should all be 0):
date                        0
buyer_name                  0
tender_title                0
mainprocurementcategory     0
tender_minvalue             0
tender_status               0
award_title                 0
award_date                  0
award_value                 0
award_supplier              0
days_to_award               0
budget_utilization_ratio    0

Column dtypes:
date                        datetime64[us]
buyer_name                             str
tender_title                           str
mainprocurementcategory                str
tender_minvalue                    float64
tender_status                          str
award_title                            str
award_date                  datetime64[us]
award_value                        float64
award_supplier                         str
days_to_award                        int64
budget_utilization_ratio           float64

Sample data:

,date,buyer_name,tender_title,mainprocurementcategory,tender_minvalue,tender_status,award_title,award_date,award_value,award_supplier,days_to_award,budget_utilization_ratio
0,2014-12-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,Goods,1.499502e+09,Complete,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,2014-12-24,1.157888e+09,Cv. Usaha Selaras,16,0.772182
1,2015-04-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/ Satwa Tmr (Buah Da...,Goods,9.039069e+09,Complete,Pengadaan Makanan Binatang/ Satwa Tmr (Buah Da...,2015-04-24,6.867945e+09,Cv. Pangan Indo,115,0.759807
2,2015-04-23,Pemerintah Daerah Provinsi Dki Jakarta,Belanja Bahan Pakan Daging/Binatang Hidup,Goods,6.271924e+09,Complete,Belanja Bahan Pakan Daging/Binatang Hidup,2015-04-23,6.196783e+09,Cv. Rosadia Petaho,113,0.988019


In [84]:
output_path = 'alldataocFinal.csv'
data.to_csv(output_path, index=False)
print(f"Exported: {output_path}")
print(f"  Rows   : {len(data)}")
print(f"  Columns: {data.columns.tolist()}")

Exported: alldataocFinal.csv
  Rows   : 8678
  Columns: ['date', 'buyer_name', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier', 'days_to_award', 'budget_utilization_ratio']


## Train / Test Split (85/15)

In [85]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    data,
    test_size=0.15,
    train_size=0.85,
    random_state=42
)

print(f"Total rows    : {len(data)}")
print(f"Training rows : {len(train_data)} ({len(train_data)/len(data)*100:.1f}%)")
print(f"Testing rows  : {len(test_data)} ({len(test_data)/len(data)*100:.1f}%)")

Total rows    : 8678
Training rows : 7376 (85.0%)
Testing rows  : 1302 (15.0%)
